# Samaritan — GRPO, standalone

No `trl`, no `unsloth`, no `mergekit`. Only **torch, transformers, peft** —
which already work in this runtime; it was always the framework wrapper that
died, never the model.

GRPO written out is two pages: sample G completions per prompt, grade them, and
set each one's advantage to how far its reward sits from its *group's* mean in
units of the group's spread. The siblings are the baseline. No critic, no value
head, no reference model.

**Runtime → Change runtime type → A100.** An L4 will be tight: bf16 weights are
~8 GB and the generation KV cache for 8 concurrent completions adds ~10 GB.

### What to expect

Rollouts are plain HF `generate` — no vLLM — so a step is **minutes, not
seconds**: 8 completions of up to 8k tokens is ~65k tokens each step. The
default 40 steps is a run that shows whether reward moves. It is not a finished
model, and it is not meant to be.

### Why it trains the BASE

Measured 2026-09-16 at a budget where both models finish: base **39/40**, the
185-trace SFT student **30/40**, McNemar p = 0.012. The student is not more
concise — it fails to *terminate*, running past 16,000 tokens on nine of forty
items where the base never exceeds 7,121. A verifiable reward scores a runaway
rollout 0, so GRPO attacks that defect directly.

## 1. Drive, and the bundle

Drive is mounted first because **checkpoints go there, not to `/content`**.
Colab preempts, and anything under `/content` dies with the runtime — an
unattended run that gets reclaimed at step 30 would otherwise leave nothing at
all. With Drive, re-running the training cell picks up where it stopped.

In [ ]:
import os, glob, subprocess, sys, time

from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/samaritan-grpo'
os.makedirs(OUT, exist_ok=True)
print('checkpoints ->', OUT)

if not glob.glob('grpo-bundle.tar.gz'):
    from google.colab import files
    print('Select grpo-bundle.tar.gz (in your models folder)...')
    files.upload()
assert os.path.exists('grpo-bundle.tar.gz'), 'bundle not uploaded'
# NOT `rm -rf samaritan` first: that deletes target/ along with it, so the
# compiled grader vanishes while the variable holding its path does not -
# and the trainer then reports a grader that returned nothing. tar overwrites
# in place, and cargo rebuilds only what actually changed.
!mkdir -p samaritan && tar -xzf grpo-bundle.tar.gz -C samaritan
print(sorted(os.listdir('samaritan/training')))

## 2. The maths, verified before anything else

Runs on CPU in a second. If the advantage sign or the length normalisation were
wrong, the model would train toward exactly what the grader rejects while the
loss curve looked perfectly healthy — so this runs first, every time.

In [ ]:
!cd samaritan && python training/test_grpo_standalone.py

## 3. The reward — the harness's own Rust grader

Not a Python reimplementation. A second oracle that silently disagrees with the
grader every measurement in this project used is how a model gets optimised
toward the wrong target without anyone noticing.

In [ ]:
t0 = time.time()
if not os.path.exists(os.path.expanduser('~/.cargo/bin/cargo')):
    subprocess.run('curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal -q',
                   shell=True, check=True)
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']

!cd samaritan && cargo build --release -q -p samaritan-corpus --example grade_batch
!cd samaritan && cargo build --release -q -p samaritan-curriculum --example generate

def built(name):
    base = f'samaritan/target/release/examples/{name}'
    hits = [base] if os.path.exists(base) else sorted(glob.glob(base + '-*'))
    hits = [h for h in hits if not h.endswith('.d')]
    assert hits, f'{name} did not build'
    return os.path.abspath(hits[0])

GRADER, GENERATE = built('grade_batch'), built('generate')
print(f'built in {time.time()-t0:.0f}s')

import json
probe = '\n'.join([
    json.dumps({'given': 'Answer: 42', 'answer': '42', 'answer_kind': 'exactMatch'}),
    json.dumps({'given': 'Answer: 43', 'answer': '42', 'answer_kind': 'exactMatch'}),
])
out = subprocess.run([GRADER], input=probe, capture_output=True, text=True)
v = [json.loads(l)['correct'] for l in out.stdout.splitlines() if l.strip()]
assert v == [True, False], f'grader is wrong: {v} / {out.stderr[:300]}'
print('grader verified:', v)

## 4. Training problems

Seeds 1003/1004/1005 — not 777/883/884, which are eval seeds. Same generator,
disjoint problems, so training cannot touch the measurement set by
construction rather than by trusting a filter.

**150 problems at each of d3, d4 and d5.** The previous run used d3 alone and
got one gradient step out of forty: the base solves generated d3 at 39/40, so
eight rollouts come back eight-correct and teach nothing. A group has to
*disagree* to carry any signal at all.

In [ ]:
# A SPREAD of difficulties. GRPO learns only from groups whose rollouts
# DISAGREE, and the base solves generated d3 at 39/40 - so eight rollouts on
# a d3 problem come back eight-correct and teach nothing. Measured: a 40-step
# d3 run produced exactly one gradient step.
#
# The uncertain band for d4/d5 is not measured, so cover the range and let the
# trainer find it - unanimous prompts are retired, so budget migrates to the
# ones that actually disagree.
import collections

TRAIN = '/content/grpo-train-mixed.jsonl'
open(TRAIN, 'w').close()
for d in ('3', '4', '5'):
    part = f'/content/gen-d{d}.jsonl'
    os.environ.update(SEED=f'100{d}', DIFFICULTY=d, COUNT='150',
                      SPLIT_LABEL='train', OUT=part)
    subprocess.run([GENERATE], check=True, capture_output=True)
    with open(TRAIN, 'a') as dst, open(part) as src:
        dst.write(src.read())

rows = [json.loads(l) for l in open(TRAIN) if l.strip()]
fam = collections.Counter(r['id'].split('-')[1] for r in rows)
dif = collections.Counter(r['id'].split('-d')[1][0] for r in rows)
print(f'{len(rows)} problems, {len(fam)} families')
print(f'  by difficulty: {dict(sorted(dif.items()))}')
print(f'  by family    : {dict(fam)}')

## 4b. How much of this set can carry a DENSE reward

Binary correct/incorrect carries no gradient when every rollout in a group
agrees, and on this curriculum they usually do. The generators record their own
solution path, so a group of eight wrong answers can still be *ranked* by how
far along that path each one got — which turns a dead group into a gradient.

That only works where a problem has intermediate quantities a solver had to
derive. Integers only, two digits or more, absent from the question, matched on
digit boundaries — so `28` is not credited for appearing inside `128`. Some
families are rich in these (`sat`, `divideconquer`); others state their answer
in one step and fall back to binary.

This asks the grader, not a copy of its rules: a second implementation of the
anchor logic would be a second oracle, and this project has already paid for
having two of those.

In [ ]:
import collections

probe_in = '\n'.join(json.dumps({
    'given': '', 'answer': r['answer'],
    'answer_kind': r.get('answer_kind', 'exactMatch'),
    'question': r['question'], 'steps': r.get('steps') or [],
}) for r in rows)
p = subprocess.run(GRADER, shell=True, input=probe_in,
                   capture_output=True, text=True)
flags = [json.loads(l).get('step_recall') is not None
         for l in p.stdout.splitlines() if l.strip()]
assert len(flags) == len(rows), f'grader returned {len(flags)} of {len(rows)}'

byfam = collections.defaultdict(lambda: [0, 0])
for r, ok in zip(rows, flags):
    f = byfam[r['id'].split('-')[1]]
    f[0] += ok; f[1] += 1
n = sum(flags)
print(f'{n}/{len(rows)} problems ({n/len(rows):.0%}) can carry partial credit')
for fam, (a, b) in sorted(byfam.items(), key=lambda kv: -kv[1][0] / kv[1][1]):
    bar = '#' * round(20 * a / b)
    print(f'  {fam:<14} {a:>3}/{b:<3} {bar}')
if n / len(rows) < 0.3:
    print('\n*** under a third: most groups will fall back to binary reward,',
          'and unanimous groups stay unanimous. Expect the step-10 abort.')

## 5. Dependencies, checked the way training uses them

`peft` is the only install; torch and transformers are already here. The check
runs in a **subprocess**, importing exactly what the trainer imports — an import
that works in this kernel proves nothing about the separate interpreter that
runs the script, which is how four earlier runs reached a model load before
dying on a missing package.

In [ ]:
!pip -q install peft 2>&1 | tail -2

PROBE = "import torch, transformers, peft\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\nfrom peft import LoraConfig, get_peft_model\nprint('  torch', torch.__version__, '| transformers', transformers.__version__,\n      '| peft', peft.__version__)\nok = torch.cuda.is_available()\nprint('  cuda', ok, '|', torch.cuda.get_device_name(0) if ok else 'NO GPU')\nif ok:\n    free, total = torch.cuda.mem_get_info()\n    print(f'  vram {free/1e9:.1f} GB free of {total/1e9:.1f} GB')\nassert ok, 'no GPU - Runtime > Change runtime type'"
r = subprocess.run([sys.executable, '-c', PROBE], capture_output=True, text=True)
print(r.stdout, end='')
if r.returncode != 0:
    raise SystemExit('training imports FAILED in a subprocess:\n' + r.stderr[-2500:])
print('\nall training imports resolve in a subprocess')

## 6. Dry run — dataset and reward path, no GPU

Proves the trainer can read the problems and reach the grader, before a model is
loaded. A reward function that silently returns 0 trains the model to do nothing,
slowly and expensively.

In [ ]:
# python -u: piping through tee makes stdout BLOCK-buffered, so output sits in
#   an 8 KB buffer instead of appearing. That turns a working run into a blank
#   cell, which reads as a hang - and did.
!cd samaritan && python -u training/grpo_standalone.py {TRAIN} \
    --grader {GRADER} --dry-run

## 7. Smoke test — 2 steps

The first run that touches the GPU. Two steps prove generation, grading and a
backward pass all work together in *this* runtime. Roughly five minutes; if it
fails, it fails here rather than an hour into the real run.

Small on purpose — 2 completions of 512 tokens — so it exercises every code path
cheaply.

**`--smoke` forces the backward pass.** The previous version generated two
truncated completions, got a flat group, skipped the gradient step and printed
success — so it proved generation and grading and never once touched the part
most likely to break, where gradient checkpointing, LoRA and completion masking
all interact. `--smoke` substitutes synthetic advantages, runs one real backward
and asserts the gradient reached the LoRA weights. It is explicitly not
learning: the numbers are invented and nothing is saved.

It starts by removing `torchao`. peft's LoRA dispatcher calls
`is_torchao_available()`, which **raises** rather than returning False when the
installed torchao predates 0.16 — Colab ships 0.10.0, whose `.so` files already
fail to load under Python 3.13. Nothing here uses it.

Then it builds a real LoRA before the 4B is touched. Cell 5 checked the imports
and that failure happened *inside* `get_peft_model`, which cell 5 never called —
the same shape of mistake as `import trl` passing while `GRPOTrainer` was
unimportable. A 1-layer llama from config has the same module names as Qwen3, so
it exercises the identical dispatcher in about a second.

In [ ]:
!pip -q uninstall -y torchao 2>&1 | tail -2

PEFT_PROBE = "from transformers import AutoConfig, AutoModelForCausalLM\nfrom peft import LoraConfig, get_peft_model\nimport peft.import_utils as iu\nprint('  torchao check:', iu.is_torchao_available())   # must return, not raise\ncfg = AutoConfig.for_model('llama', hidden_size=16, intermediate_size=32,\n                           num_hidden_layers=1, num_attention_heads=2,\n                           num_key_value_heads=2, vocab_size=64)\nm = get_peft_model(\n    AutoModelForCausalLM.from_config(cfg),\n    LoraConfig(r=4, lora_alpha=4, lora_dropout=0.0, bias='none',\n               task_type='CAUSAL_LM',\n               target_modules=['q_proj','k_proj','v_proj','o_proj',\n                               'gate_proj','up_proj','down_proj']),\n)\nn = sum(p.numel() for p in m.parameters() if p.requires_grad)\nassert n > 0, 'LoRA wrapped but nothing is trainable'\nprint(f'  get_peft_model OK - {n} trainable parameters on a toy model')"
r = subprocess.run([sys.executable, '-c', PEFT_PROBE], capture_output=True, text=True)
print(r.stdout, end='')
if r.returncode != 0:
    raise SystemExit('LoRA wrapping still fails - fix before the GPU:\n' + r.stderr[-2500:])
print()
print('LoRA path verified; the 4B will wrap the same way')

!cd samaritan && python -u training/grpo_standalone.py {TRAIN} \
    --grader {GRADER} --smoke --generations 2 --max-new 512 \
    --save-every 0 --output /content/adapters/smoke

## 7b. Probe — does the model actually disagree with itself?

**This is the go/no-go, and it costs about twenty minutes.** Everything up to
here is offline reasoning about whether partial credit *can* work: the problems
carry scoreable intermediates, the maths is tested, the anchors reject an
off-by-one probe. None of it proves that eight real rollouts differ from each
other — every test used completions written by hand, which differ by
construction.

So: generate 6 groups at full length, grade them, print the spread, train on
nothing. Three outcomes, and each one says what to do:

- **most groups carry a gradient, some contain a correct answer** — start the run
- **under a third carry a gradient** — the real run would abort at step 10 anyway;
  changing `DIFFICULTY` is cheaper than finding out an hour in
- **nothing is ever correct** — partial credit would train it toward reaching
  intermediates and never toward finishing, because nothing has shown it what
  finishing looks like. Mix in an easier difficulty first.

In [ ]:
# No backward pass, no optimiser, nothing written. Six groups of eight at the
# full 8k budget, which is the same generation the real run does.
!cd samaritan && python -u training/grpo_standalone.py {TRAIN} \
    --grader {GRADER} --probe 6 --generations 8 --max-new 8192 \
    --output /content/adapters/unused 2>&1 | tee {OUT}/probe.log

## 8. Train

**Watch the `reward` column, not the loss.** If mean reward climbs over the first
~15 steps, RL is working.

The log now prints **`n/8 right`** as well as mean reward, and they are different
numbers on purpose: with `--shaping` a group can move the policy while getting
nothing right, which is the whole point. Partial credit applies only to wrong
completions, and is capped well below 1, so a wrong answer can never outscore a
right one — it reorders failures, it does not compete with success.

Around step 10 a reward-health line prints. If it reports no gradient it also says
*which* cause, because they need opposite fixes: rollouts with no `Answer:` line
are being cut off (raise `--max-new`), while rollouts that finish and still agree
mean the problems are solved-or-doomed (shaping, or a different `DIFFICULTY`).
Neither is fixed by training longer.

Checkpoints land in Drive every 5 steps and the log is tee'd there too, so
both survive the runtime. If Colab reclaims the session, re-run this one cell:
`--resume` continues the adapter rather than starting over.

Two things now stop a wasted night rather than reporting one:

- **a flat reward aborts** at step 10 instead of spending the rest of the run on
  a zero gradient, and prints which of the two causes it was
- **CUDA OOM halves the group and retries** rather than killing the run — a
  smaller group still teaches something; a crash at step 3 teaches nothing

In [ ]:
# --resume: re-running this cell after a preemption continues the adapter
#   already in Drive rather than starting over from random weights.
# 2>&1 | tee: the log lands in Drive too, so the outcome is readable from a
#   phone even if the runtime is long gone.
!cd samaritan && python -u training/grpo_standalone.py {TRAIN} \
    --grader {GRADER} --steps 40 --generations 8 --max-new 8192 \
    --shaping 0.25 --save-every 5 --resume \
    --output {OUT}/reasoning-grpo 2>&1 | tee -a {OUT}/train.log

## 9. Result

Everything is already in Drive - nothing here needs you at the keyboard. This
just prints what landed, so the outcome is legible when you come back.

In [ ]:
adapter = f'{OUT}/reasoning-grpo'
print('adapter:', adapter)
for f in sorted(glob.glob(adapter + '/*')):
    print(f'  {os.path.basename(f):<28} {os.path.getsize(f)/1e6:>8.2f} MB')

# The reward trace is the result. Flat means it learned nothing, whatever the
# loss did.
log = f'{OUT}/train.log'
if os.path.exists(log):
    steps = [l for l in open(log, encoding='utf-8', errors='replace')
             if l.startswith('step')]
    print(f'\n{len(steps)} steps logged. First and last five:')
    for l in steps[:5] + (['  ...'] if len(steps) > 10 else []) + steps[-5:]:
        print('  ' + l.rstrip())

print('\nDownload it whenever you are back:')
print('  from google.colab import files')
print(f'  files.download(\'{OUT}/train.log\')')

## 10. Did it help? — base and trained, same problems, same runtime

The adapter on its own is not a result. The 39/40 base figure came from Ollama
at Q4 and temperature 0.6, so an HF bf16 number is not comparable to it — the
only honest comparison is base against trained **here**, on the same held-out
problems, in the same runtime. LoRA makes that cheap: `disable_adapter()` IS
the base model, same weights, same kernels, same sampling.

Held out by seed. 883/884 are the eval seeds and were never generated into the
training file, so this is disjoint by construction rather than by a filter.

**Paired, so each problem is its own control.** What matters is not the two
totals but the disagreements: how many the adapter fixed against how many it
broke. With a handful of items either number is noise — treat a 1–2 problem
difference as nothing. Roughly two minutes per problem per model, so the
default 40 is about two and a half hours; raise it only if the run has already
shown a reward trend worth measuring.

In [ ]:
N_EVAL = 40

# The trainer's own default, read from the script rather than retyped here -
# an eval against a different base is not an eval.
BASE = 'Qwen/Qwen3-4B-Thinking-2507'

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

os.environ.update(SEED='883', DIFFICULTY='4', COUNT=str(N_EVAL),
                  SPLIT_LABEL='eval', OUT='/content/eval-d4.jsonl')
subprocess.run([GENERATE], check=True, capture_output=True)
ev = [json.loads(l) for l in open('/content/eval-d4.jsonl') if l.strip()]
print(f'{len(ev)} held-out problems (seed 883, never in training)')

sys.path.insert(0, '/content/samaritan/training')
from grpo_standalone import REASONING_SYSTEM, grade

tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
tok.padding_side = 'left'
base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.bfloat16,
                                            device_map='cuda')
model = PeftModel.from_pretrained(base, f'{OUT}/reasoning-grpo')
model.eval()

def answer(row, with_adapter):
    chat = [{'role': 'system', 'content': REASONING_SYSTEM},
            {'role': 'user', 'content': row['question']}]
    p = tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    enc = tok(p, return_tensors='pt', truncation=True, max_length=2048).to('cuda')
    # Same seed for both arms on each problem, so a disagreement is the
    # adapter and not the sampler.
    torch.manual_seed(hash(row['id']) % (2**31))
    def gen():
        with torch.no_grad():
            o = model.generate(**enc, max_new_tokens=8192, do_sample=True,
                               temperature=0.6, top_p=0.95,
                               pad_token_id=tok.pad_token_id)
        return tok.decode(o[0][enc.input_ids.shape[1]:], skip_special_tokens=True)
    if with_adapter:
        return gen()
    with model.disable_adapter():
        return gen()

fixed, broke, both_ok, both_bad = [], [], 0, 0
for i, row in enumerate(ev, 1):
    b = grade(GRADER, [(answer(row, False), row)])[0][0]
    t = grade(GRADER, [(answer(row, True), row)])[0][0]
    if b and t: both_ok += 1
    elif not b and not t: both_bad += 1
    elif t: fixed.append(row['id'])
    else: broke.append(row['id'])
    rid = row['id']
    print(f'  {i}/{len(ev)} {rid:<26} base {int(b)} trained {int(t)}',
          flush=True)

n = len(ev)
print(f'\nbase    {both_ok + len(broke)}/{n}')
print(f'trained {both_ok + len(fixed)}/{n}')
print(f'\n  both right {both_ok}   both wrong {both_bad}')
print(f'  FIXED by the adapter: {len(fixed)}  {fixed}')
print(f'  BROKEN by the adapter: {len(broke)}  {broke}')
d = len(fixed) + len(broke)
if d < 5:
    print(f'\nonly {d} problems disagreed - too few to call either way. This is',
          'the noise floor, not a result.')
elif len(fixed) > len(broke):
    print(f'\n{len(fixed)} fixed against {len(broke)} broken, out of {d} that',
          'disagreed. Worth a bigger eval.')
else:
    print(f'\n{len(broke)} broken against {len(fixed)} fixed. The adapter is not',
          'helping - do not merge it.')

## Then

It is a LoRA adapter, not a GGUF — merge it into the base and convert before
serving through Ollama, then A/B it against the base with the settings that
finally produced a clean measurement:

```
-Limit 40 -Tag "-grpo" -Shards 3      # MaxTokens/Ctx/Seed are correct by default
```

The bar is the base's **39/40**. Anything that does not clear it is not progress,
however good the reward curve looked.